# extractrs vs xarray-spatial — Zonal Statistics Benchmark

This notebook benchmarks **extractrs** (Rust/PyO3) against **xarray-spatial** (Numba/CuPy) for zonal statistics on synthetic raster data.

**Key difference in approach:**
- **xarray-spatial**: Rasterizes polygons to a zone grid (each cell → one zone ID, no fractional coverage), then aggregates per zone via argsort + Python loop. CuPy path accelerates the sort/filter on GPU but still loops zones on CPU.
- **extractrs (exact)**: Computes exact subpixel coverage fractions per polygon/cell intersection in Rust, then applies coverage-weighted statistics.
- **extractrs (center)**: Uses cell-center point-in-polygon (same model as xarray-spatial), but in Rust. Should produce identical results to xarray-spatial.

**Hardware:**
- CPU: host system
- GPU: NVIDIA RTX 4080 SUPER (16 GB)

We test five configurations:
1. **xarray-spatial (CPU, GeoDataFrame)** — full user-facing API (includes internal rasterization)
2. **xarray-spatial (CPU, pre-rasterized)** — best-case CPU with zones already rasterized
3. **xarray-spatial (GPU/CuPy, pre-rasterized)** — CuPy-backed arrays on GPU
4. **extractrs (exact)** — vector polygons → exact fractional coverage → weighted stats
5. **extractrs (center)** — vector polygons → cell-center PIP → binary stats (apples-to-apples with xarray-spatial)

In [1]:
import time
import warnings

import cupy as cp
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
from shapely.geometry import box, Polygon
from shapely.affinity import rotate

from xrspatial.zonal import stats as xrs_stats
import extractrs  # registers .extrs accessor

## Test Data Generation

Create synthetic raster data and two types of zone polygons:
- **Axis-aligned rectangles** — tile the grid perfectly (no partial cells)
- **Rotated polygons** — create fractional cell coverage at boundaries

In [2]:
def make_raster(ny, nx):
    """Create a 2D DataArray with random values and cell-center coords."""
    rng = np.random.default_rng(42)
    y = np.arange(ny, dtype=np.float64) + 0.5
    x = np.arange(nx, dtype=np.float64) + 0.5
    data = rng.standard_normal((ny, nx)).astype(np.float64)
    return xr.DataArray(data, dims=["y", "x"], coords={"y": y, "x": x})


def make_rect_zones(ny, nx, nzy, nzx):
    """Create axis-aligned rectangular zones that tile the grid."""
    zone_h, zone_w = ny / nzy, nx / nzx
    polys, ids = [], []
    zid = 1
    for iy in range(nzy):
        for ix in range(nzx):
            polys.append(box(ix * zone_w, iy * zone_h, (ix + 1) * zone_w, (iy + 1) * zone_h))
            ids.append(zid)
            zid += 1
    return gpd.GeoDataFrame({"zone_id": ids}, geometry=polys)


def make_rotated_zones(ny, nx, nzy, nzx, angle=15):
    """Create rotated rectangular zones — partial cell coverage at boundaries."""
    zone_h, zone_w = ny / nzy, nx / nzx
    polys, ids = [], []
    zid = 1
    for iy in range(nzy):
        for ix in range(nzx):
            cx = (ix + 0.5) * zone_w
            cy = (iy + 0.5) * zone_h
            rect = box(cx - zone_w * 0.4, cy - zone_h * 0.4,
                        cx + zone_w * 0.4, cy + zone_h * 0.4)
            rotated = rotate(rect, angle, origin=(cx, cy))
            polys.append(rotated)
            ids.append(zid)
            zid += 1
    return gpd.GeoDataFrame({"zone_id": ids}, geometry=polys)


def rasterize_zones(gdf, da):
    """Rasterize a GeoDataFrame of zones onto the same grid as da.

    Each cell gets the zone_id of the polygon whose interior contains
    the cell center. Cells outside all polygons get 0.
    """
    from rasterio.features import rasterize as rio_rasterize
    from rasterio.transform import from_origin

    y = da["y"].values
    x = da["x"].values
    dy = abs(float(y[1] - y[0]))
    dx = abs(float(x[1] - x[0]))

    # Cell-edge transform: top-left corner
    transform = from_origin(float(x.min()) - dx / 2, float(y.max()) + dy / 2, dx, dy)

    shapes = [(geom, zid) for geom, zid in zip(gdf.geometry, gdf["zone_id"])]
    zone_arr = rio_rasterize(
        shapes, out_shape=(len(y), len(x)), transform=transform,
        fill=0, dtype=np.int32, all_touched=False,
    )
    return xr.DataArray(zone_arr, dims=["y", "x"], coords={"y": y, "x": x})

## Benchmark Harness

In [3]:
def bench_xrs_gdf(da, gdf, stat="mean"):
    """xarray-spatial with GeoDataFrame input (rasterizes internally)."""
    t0 = time.perf_counter()
    result = xrs_stats(zones=gdf, values=da, stats_funcs=[stat], column="zone_id")
    return result, time.perf_counter() - t0


def bench_xrs_rasterized(da, zones_da, stat="mean"):
    """xarray-spatial with pre-rasterized numpy zones."""
    t0 = time.perf_counter()
    result = xrs_stats(zones=zones_da, values=da, stats_funcs=[stat])
    return result, time.perf_counter() - t0


def bench_xrs_gpu(da, zones_da, stat="mean"):
    """xarray-spatial with CuPy-backed arrays on GPU."""
    # Transfer to GPU
    values_gpu = xr.DataArray(cp.asarray(da.values), dims=da.dims, coords=da.coords)
    zones_gpu = xr.DataArray(cp.asarray(zones_da.values), dims=zones_da.dims, coords=zones_da.coords)
    cp.cuda.Stream.null.synchronize()  # ensure transfer is done

    t0 = time.perf_counter()
    result = xrs_stats(zones=zones_gpu, values=values_gpu, stats_funcs=[stat])
    cp.cuda.Stream.null.synchronize()
    return result, time.perf_counter() - t0


def bench_extractrs(da, gdf, stat="mean", method="exact"):
    """extractrs with GeoDataFrame input."""
    t0 = time.perf_counter()
    result = da.extrs.zonal_stats(gdf, stat=stat, id_col="zone_id", method=method)
    return result, time.perf_counter() - t0

## Warmup

Run each backend once on small data to trigger Numba JIT compilation and CuPy kernel caching.

In [4]:
da_warm = make_raster(50, 50)
gdf_warm = make_rect_zones(50, 50, 3, 3)
zones_warm = rasterize_zones(gdf_warm, da_warm)

# Warmup all paths
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _ = bench_xrs_gdf(da_warm, gdf_warm)
    _ = bench_xrs_rasterized(da_warm, zones_warm)
    _ = bench_xrs_gpu(da_warm, zones_warm)
    _ = bench_extractrs(da_warm, gdf_warm, method="exact")
    _ = bench_extractrs(da_warm, gdf_warm, method="center")

print("Warmup complete.")

Warmup complete.


## Benchmark — Axis-Aligned Zones (Scaling Test)

Rectangle zones that tile the grid perfectly. Tests pure computation speed across grid sizes.

In [5]:
configs = [
    # (ny, nx, n_zones_y, n_zones_x)
    (100, 100, 5, 5),        # 10K cells, 25 zones
    (500, 500, 10, 10),      # 250K cells, 100 zones
    (1000, 1000, 20, 20),    # 1M cells, 400 zones
    (2000, 2000, 25, 25),    # 4M cells, 625 zones
    (4000, 4000, 40, 40),    # 16M cells, 1600 zones
]

rows = []
for ny, nx, nzy, nzx in configs:
    n_cells = ny * nx
    n_zones = nzy * nzx
    print(f"--- {ny}x{nx} grid, {n_zones} zones ({n_cells:,} cells) ---")

    da = make_raster(ny, nx)
    gdf = make_rect_zones(ny, nx, nzy, nzx)
    zones_da = rasterize_zones(gdf, da)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        _, t_ext_exact = bench_extractrs(da, gdf, method="exact")
        print(f"  extractrs (exact):      {t_ext_exact:8.4f}s")

        _, t_ext_center = bench_extractrs(da, gdf, method="center")
        print(f"  extractrs (center):     {t_ext_center:8.4f}s")

        _, t_xrs_gdf = bench_xrs_gdf(da, gdf)
        print(f"  xrspatial (CPU, GDF):   {t_xrs_gdf:8.4f}s")

        _, t_xrs_rast = bench_xrs_rasterized(da, zones_da)
        print(f"  xrspatial (CPU, rast):  {t_xrs_rast:8.4f}s")

        _, t_xrs_gpu = bench_xrs_gpu(da, zones_da)
        print(f"  xrspatial (GPU, rast):  {t_xrs_gpu:8.4f}s")

    print()
    rows.append({
        "grid": f"{ny}x{nx}",
        "cells": n_cells,
        "zones": n_zones,
        "extrs_exact": t_ext_exact,
        "extrs_center": t_ext_center,
        "xrs_cpu_gdf": t_xrs_gdf,
        "xrs_cpu_rast": t_xrs_rast,
        "xrs_gpu_rast": t_xrs_gpu,
    })

--- 100x100 grid, 25 zones (10,000 cells) ---
  extractrs (exact):        0.0005s
  extractrs (center):       0.0005s
  xrspatial (CPU, GDF):     0.0017s
  xrspatial (CPU, rast):    0.0005s
  xrspatial (GPU, rast):    0.0019s

--- 500x500 grid, 100 zones (250,000 cells) ---
  extractrs (exact):        0.0024s
  extractrs (center):       0.0029s
  xrspatial (CPU, GDF):     0.0188s
  xrspatial (CPU, rast):    0.0116s
  xrspatial (GPU, rast):    0.0054s

--- 1000x1000 grid, 400 zones (1,000,000 cells) ---
  extractrs (exact):        0.0104s
  extractrs (center):       0.0123s
  xrspatial (CPU, GDF):     0.0815s


  xrspatial (CPU, rast):    0.0535s
  xrspatial (GPU, rast):    0.0146s

--- 2000x2000 grid, 625 zones (4,000,000 cells) ---
  extractrs (exact):        0.0372s
  extractrs (center):       0.0430s


  xrspatial (CPU, GDF):     0.5123s


  xrspatial (CPU, rast):    0.3626s
  xrspatial (GPU, rast):    0.0225s

--- 4000x4000 grid, 1600 zones (16,000,000 cells) ---


  extractrs (exact):        0.1643s


  extractrs (center):       0.2309s


  xrspatial (CPU, GDF):     1.7724s


  xrspatial (CPU, rast):    1.0554s
  xrspatial (GPU, rast):    0.0684s



In [6]:
df = pd.DataFrame(rows)
df["center_vs_cpu_gdf"] = df["xrs_cpu_gdf"] / df["extrs_center"]
df["center_vs_cpu_rast"] = df["xrs_cpu_rast"] / df["extrs_center"]
df["center_vs_gpu"] = df["xrs_gpu_rast"] / df["extrs_center"]

print("=" * 100)
print("RESULTS — Axis-Aligned Zones (stat=mean)")
print("=" * 100)
print(df[["grid", "cells", "zones", "extrs_exact", "extrs_center",
          "xrs_cpu_rast", "xrs_gpu_rast",
          "center_vs_cpu_rast", "center_vs_gpu"]].to_string(
    index=False,
    float_format="{:.4f}".format,
    col_space=12,
))
print()
print("center_vs_cpu_rast = xrspatial CPU time / extractrs center time")
print("center_vs_gpu      = xrspatial GPU time / extractrs center time")

RESULTS — Axis-Aligned Zones (stat=mean)
        grid        cells        zones  extrs_exact  extrs_center  xrs_cpu_rast  xrs_gpu_rast  center_vs_cpu_rast  center_vs_gpu
     100x100        10000           25       0.0005        0.0005        0.0005        0.0019              1.0468         4.0097
     500x500       250000          100       0.0024        0.0029        0.0116        0.0054              3.9891         1.8628
   1000x1000      1000000          400       0.0104        0.0123        0.0535        0.0146              4.3583         1.1862
   2000x2000      4000000          625       0.0372        0.0430        0.3626        0.0225              8.4298         0.5224
   4000x4000     16000000         1600       0.1643        0.2309        1.0554        0.0684              4.5709         0.2961

center_vs_cpu_rast = xrspatial CPU time / extractrs center time
center_vs_gpu      = xrspatial GPU time / extractrs center time


## Accuracy Validation — Rotated Zones

Rotated polygons create fractional cell coverage at boundaries.

**Two comparisons:**
1. **extractrs (center) vs xarray-spatial** — same algorithm, should be identical (validates correctness)
2. **extractrs (exact) vs xarray-spatial** — different algorithms, quantifies the accuracy gap from cell-center approximation

In [7]:
# 1000x1000 grid, 400 rotated zones
ny, nx, nzy, nzx = 1000, 1000, 20, 20
da = make_raster(ny, nx)
gdf_rot = make_rotated_zones(ny, nx, nzy, nzx, angle=15)

zones_rot = rasterize_zones(gdf_rot, da)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    xrs_result, t_xrs = bench_xrs_rasterized(da, zones_rot)
    xrs_gpu_result, t_xrs_gpu = bench_xrs_gpu(da, zones_rot)
    ext_exact_result, t_ext_exact = bench_extractrs(da, gdf_rot, method="exact")
    ext_center_result, t_ext_center = bench_extractrs(da, gdf_rot, method="center")

print(f"Timing (1000x1000, 400 rotated zones):")
print(f"  extractrs (exact):      {t_ext_exact:.4f}s")
print(f"  extractrs (center):     {t_ext_center:.4f}s")
print(f"  xrspatial (CPU, rast):  {t_xrs:.4f}s  ({t_xrs/t_ext_center:.1f}x vs center)")
print(f"  xrspatial (GPU, rast):  {t_xrs_gpu:.4f}s  ({t_xrs_gpu/t_ext_center:.1f}x vs center)")

Timing (1000x1000, 400 rotated zones):
  extractrs (exact):      0.0288s
  extractrs (center):     0.0110s
  xrspatial (CPU, rast):  0.0787s  (7.2x vs center)
  xrspatial (GPU, rast):  0.0129s  (1.2x vs center)


In [8]:
# xrspatial means (from pre-rasterized CPU run)
xrs_means = xrs_result.set_index("zone")["mean"].sort_index()
xrs_means = xrs_means[xrs_means.index != 0]

# extractrs center means
ext_center_means = ext_center_result.to_series().sort_index()

# extractrs exact means
ext_exact_means = ext_exact_result.to_series().sort_index()

# --- Validation 1: extractrs center vs xarray-spatial (should be identical) ---
common = xrs_means.index.intersection(ext_center_means.index)
diff_center = np.abs(xrs_means.loc[common].values - ext_center_means.loc[common].values)
valid = ~np.isnan(diff_center)

print("=" * 60)
print("VALIDATION: extractrs (center) vs xarray-spatial")
print("=" * 60)
print(f"  Zones compared:  {valid.sum()}")
print(f"  Max abs diff:    {diff_center[valid].max():.2e}")
print(f"  Mean abs diff:   {diff_center[valid].mean():.2e}")
print(f"  All < 1e-14:     {(diff_center[valid] < 1e-14).all()}")
print()

# --- Comparison 2: extractrs exact vs xarray-spatial (quantify accuracy gap) ---
common2 = xrs_means.index.intersection(ext_exact_means.index)
diff_exact = xrs_means.loc[common2].values - ext_exact_means.loc[common2].values

print("=" * 60)
print("ACCURACY GAP: extractrs (exact) vs xarray-spatial")
print("=" * 60)
print(f"  Zones compared:  {len(common2)}")
print(f"  Mean abs diff:   {np.abs(diff_exact).mean():.6f}")
print(f"  Max abs diff:    {np.abs(diff_exact).max():.6f}")
print(f"  Exact matches:   {(np.abs(diff_exact) == 0).sum()}/{len(common2)}")
print()
print("The exact method accounts for fractional pixel coverage at polygon")
print("boundaries; the center/xarray-spatial method does not.")

VALIDATION: extractrs (center) vs xarray-spatial
  Zones compared:  400
  Max abs diff:    2.22e-16
  Mean abs diff:   2.44e-17
  All < 1e-14:     True

ACCURACY GAP: extractrs (exact) vs xarray-spatial
  Zones compared:  400
  Mean abs diff:   0.001812
  Max abs diff:    0.007361
  Exact matches:   0/400

The exact method accounts for fractional pixel coverage at polygon
boundaries; the center/xarray-spatial method does not.


## Summary

| Aspect | xarray-spatial | extractrs (center) | extractrs (exact) |
|--------|---------------|--------------------|--------------------|
| **Language** | Python + Numba JIT | Rust (PyO3) | Rust (PyO3) |
| **GPU support** | CuPy (sort/filter on GPU) | No | No |
| **Coverage model** | Cell-center PIP | Cell-center PIP | Exact subpixel fractions |
| **Accuracy** | Whole-pixel approx | **Identical to xrspatial** | Matches exactextract |
| **Caching** | None | Reusable across timesteps | Reusable across timesteps |
| **Use case** | Quick estimates, large grids | Drop-in faster replacement | Scientific accuracy |